In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
DATA_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

nvidia_gpu = pd.read_csv(
    DATA_PATH + "nvidia_gpu_cleaned.csv"
)

nvidia_products = pd.read_csv(
    DATA_PATH + "nvidia_product_data.csv"
)

nvidia_financial = pd.read_csv(
    DATA_PATH + "nvidia_financial_data.csv"
)

nvidia_stock = pd.read_csv(
    DATA_PATH + "nvidia_stock_features.csv"
)

nvidia_news = pd.read_csv(
    DATA_PATH + "nvidia_news_features.csv"
)

nvidia_chip_market = pd.read_csv(
    DATA_PATH + "nvidia_ai_chip_market_complete.csv"
)

amd_stock = pd.read_csv(
    DATA_PATH + "amd_stock_features.csv"
)

intel_stock = pd.read_csv(
    DATA_PATH + "intel_stock_features.csv"
)

competitor_stock = pd.read_csv(
    DATA_PATH + "competitor_stock_features.csv"
)

competitor_comparison = pd.read_csv(
    DATA_PATH + "nvidia_competitor_comparison.csv"
)

financial_model_df = pd.read_csv(
    DATA_PATH + "nvidia_financial_model_data.csv"
)

risk_df = pd.read_csv(
    DATA_PATH + "nvidia_risk_labeled_data.csv"
)

research_df = pd.read_csv(
    DATA_PATH + "nvidia_research_topics.csv"
)

valuation_input_df = pd.read_csv(
    DATA_PATH + "nvidia_valuation_data.csv"
)

In [3]:
stock_model = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_final_model.pkl"
)

stock_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_scaler.pkl"
)

sentiment_model = joblib.load(
    OUTPUT_PATH + "nvidia_financial_sentiment_model.pkl"
)

sentiment_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_sentiment_tfidf_vectorizer.pkl"
)

risk_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_risk_model.pkl"
)

risk_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_risk_tfidf_vectorizer.pkl"
)

market_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_market_competitor_model.pkl"
)

market_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_market_competitor_scaler.pkl"
)

In [4]:
research_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_research_tfidf_vectorizer.pkl"
)

research_topic_model = joblib.load(
    OUTPUT_PATH + "nvidia_research_topic_model.pkl"
)

In [5]:
print("Datasets loaded:")
print("NVIDIA GPU:", nvidia_gpu.shape)
print("NVIDIA Financial:", nvidia_financial.shape)
print("NVIDIA Stock:", nvidia_stock.shape)
print("NVIDIA News:", nvidia_news.shape)
print("NVIDIA Research:", research_df.shape)
print("NVIDIA Risk:", risk_df.shape)
print("NVIDIA Chip Market:", nvidia_chip_market.shape)

print("\nModels loaded successfully.")

Datasets loaded:
NVIDIA GPU: (1434, 103)
NVIDIA Financial: (260, 7)
NVIDIA Stock: (6308, 11)
NVIDIA News: (310, 8)
NVIDIA Research: (1953, 3)
NVIDIA Risk: (154, 4)
NVIDIA Chip Market: (85, 19)

Models loaded successfully.


In [6]:
research_df.head(10)

,text,Topic,Topic_Name
0,NVIDIA 10-K\n\nnvda-20260125 0001045810 2026 F...,4,Risk
1,20549 ________________________________________...,0,Business
2,Employer incorporation or organization) Identi...,0,Business
3,Yes ☒ No ☐ Indicate by check mark if the regis...,0,Business
4,Yes ☐ No ☒ Indicate by check mark whether the ...,0,Business
5,Yes ☒ No ☐ Indicate by check mark whether the ...,0,Business
6,Yes ☒ No ☐ Indicate by check mark whether the ...,0,Business
7,See the definitions of �large accelerated file...,0,Business
8,Large accelerated filer ☒ Accelerated filer ☐ ...,0,Business
9,☐ Indicate by check mark whether the registran...,0,Business


In [7]:
research_df["Topic"].value_counts().sort_index()

Topic
0    909
1    174
2    231
3     63
4    178
5    398
Name: count, dtype: int64

In [8]:
research_df["Topic_Name"].value_counts()

Topic_Name
Business      909
Financial     398
Market        231
Risk          178
Products      174
Technology     63
Name: count, dtype: int64

In [9]:
research_df[
    [
        "Topic",
        "Topic_Name",
        "text"
    ]
].head(20)

,Topic,Topic_Name,text
0,4,Risk,NVIDIA 10-K\n\nnvda-20260125 0001045810 2026 F...
1,0,Business,20549 ________________________________________...
2,0,Business,Employer incorporation or organization) Identi...
3,0,Business,Yes ☒ No ☐ Indicate by check mark if the regis...
4,0,Business,Yes ☐ No ☒ Indicate by check mark whether the ...
5,0,Business,Yes ☒ No ☐ Indicate by check mark whether the ...
6,0,Business,Yes ☒ No ☐ Indicate by check mark whether the ...
7,0,Business,See the definitions of �large accelerated file...
8,0,Business,Large accelerated filer ☒ Accelerated filer ☐ ...
9,0,Business,☐ Indicate by check mark whether the registran...


In [10]:
research_topic_summary = (
    research_df
    .groupby(["Topic", "Topic_Name"])
    .size()
    .reset_index(name="Document_Count")
)

research_topic_summary

,Topic,Topic_Name,Document_Count
0,0,Business,909
1,1,Products,174
2,2,Market,231
3,3,Technology,63
4,4,Risk,178
5,5,Financial,398


In [11]:
research_topic_summary.to_csv(
    DATA_PATH + "research_topic_summary_final.csv",
    index=False
)

In [12]:
terms = research_vectorizer.get_feature_names_out()

topic_keywords = {}

for topic_id, center in enumerate(
    research_topic_model.cluster_centers_
):
    top_indices = center.argsort()[-15:][::-1]

    topic_keywords[topic_id] = [
        terms[index]
        for index in top_indices
    ]

topic_keywords

{0: ['securities',
  'based',
  'financial',
  'tax',
  'stock',
  'information',
  'including',
  'statements',
  'income',
  'subject',
  'certain',
  'assets',
  'nvidia',
  'security',
  'company'],
 1: ['fiscal',
  'year',
  'fiscal year',
  '2026',
  '2027',
  'year 2026',
  'quarter',
  'half fiscal',
  'half',
  'year 2027',
  'second',
  'fiscal years',
  'years',
  'million',
  'billion'],
 2: ['ai',
  'data',
  'center',
  'data center',
  'computing',
  'software',
  'platforms',
  'infrastructure',
  'platform',
  'nvidia',
  'applications',
  'accelerated',
  'gpus',
  'customers',
  'accelerated computing'],
 3: ['form 10',
  'form',
  'report form',
  '10',
  'report',
  'annual report',
  'annual',
  'statements',
  'financial statements',
  'consolidated financial',
  'consolidated',
  'refer',
  'notes',
  'item',
  'information'],
 4: ['2026',
  '25',
  '25 2026',
  '26',
  'january',
  'billion',
  'january 25',
  '26 2026',
  '2025',
  'july 26',
  'july',
  'jan'

In [13]:
research_topics = pd.DataFrame({
    "Topic": list(topic_keywords.keys()),
    "Keywords": [
        ", ".join(words)
        for words in topic_keywords.values()
    ]
})

research_topics

,Topic,Keywords
0,0,"securities, based, financial, tax, stock, info..."
1,1,"fiscal, year, fiscal year, 2026, 2027, year 20..."
2,2,"ai, data, center, data center, computing, soft..."
3,3,"form 10, form, report form, 10, report, annual..."
4,4,"2026, 25, 25 2026, 26, january, billion, janua..."
5,5,"products, business, impact, results, customers..."


In [14]:
research_topics.to_csv(
    DATA_PATH + "nvidia_research_topic_keywords.csv",
    index=False
)

In [15]:
def predict_research_topic(
    texts,
    vectorizer,
    model
):
    vectors = vectorizer.transform(texts)

    predictions = model.predict(vectors)

    results = pd.DataFrame({
        "Text": texts,
        "Topic": predictions
    })

    return results

In [16]:
sample_research = [
    "NVIDIA develops advanced GPUs for artificial intelligence",
    "NVIDIA faces restrictions related to semiconductor exports",
    "NVIDIA reports strong growth in data center revenue"
]

research_test_results = predict_research_topic(
    sample_research,
    research_vectorizer,
    research_topic_model
)

research_test_results

,Text,Topic
0,NVIDIA develops advanced GPUs for artificial i...,0
1,NVIDIA faces restrictions related to semicondu...,0
2,NVIDIA reports strong growth in data center re...,2


In [17]:
joblib.dump(
    research_topic_model,
    OUTPUT_PATH + "nvidia_research_topic_model.pkl"
)

joblib.dump(
    research_vectorizer,
    OUTPUT_PATH + "nvidia_research_tfidf_vectorizer.pkl"
)

['../outputs/nvidia_research_tfidf_vectorizer.pkl']

In [19]:
with open(
    OUTPUT_PATH + "model_registry.json",
    "r",
    encoding="utf-8"
) as f:
    model_registry = json.load(f)

In [20]:
model_registry["Research"] = {
    "model": "TF-IDF + K-Means Topic Model",
    "path": OUTPUT_PATH + "nvidia_research_topic_model.pkl"
}

with open(
    OUTPUT_PATH + "model_registry.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_registry,
        f,
        indent=4
    )

In [21]:
for name in model_registry:
    print(name)

Stock Prediction
News Sentiment
Risk Classification
Market & Competitor
Valuation
Research


In [22]:
stock_results = pd.read_csv(
    DATA_PATH + "final_stock_model_info.csv"
)

sentiment_results = pd.read_csv(
    DATA_PATH + "sentiment_model_results.csv"
)

risk_results = pd.read_csv(
    DATA_PATH + "risk_model_results.csv"
)

market_results = pd.read_csv(
    DATA_PATH + "final_market_model_results.csv"
)

valuation_results = pd.read_csv(
    DATA_PATH + "final_valuation_metrics.csv"
)

financial_results = pd.read_csv(
    DATA_PATH + "final_financial_model_comparison.csv"
)

In [23]:
latest_stock_prediction = pd.read_csv(
    OUTPUT_PATH + "final_nvidia_stock_prediction.csv"
)

stock_signal = (
    latest_stock_prediction["Prediction"].iloc[0]
)

In [24]:
latest_market_prediction = pd.read_csv(
    OUTPUT_PATH + "latest_nvidia_market_prediction.csv"
)

market_signal = (
    latest_market_prediction["Market_Leadership"].iloc[0]
)

In [25]:
stock_accuracy = float(
    stock_results["Accuracy"].iloc[0]
)

sentiment_accuracy = float(
    sentiment_results["Accuracy"].iloc[0]
)

risk_accuracy = float(
    risk_results["Accuracy"].iloc[0]
)

market_accuracy = float(
    market_results["Accuracy"].iloc[0]
)

financial_accuracy = float(
    financial_results["Accuracy"].iloc[0]
)

In [26]:
average_growth = float(
    valuation_results.loc[
        valuation_results["Metric"]
        == "Average Revenue Growth",
        "Value"
    ].iloc[0]
)

profit_margin = float(
    valuation_results.loc[
        valuation_results["Metric"]
        == "Latest Profit Margin",
        "Value"
    ].iloc[0]
)

dcf_value = float(
    valuation_results.loc[
        valuation_results["Metric"]
        == "DCF Enterprise Value",
        "Value"
    ].iloc[0]
)

In [27]:
synthesis_features = pd.DataFrame({
    "Stock_Accuracy": [stock_accuracy],
    "Sentiment_Accuracy": [sentiment_accuracy],
    "Risk_Accuracy": [risk_accuracy],
    "Market_Accuracy": [market_accuracy],
    "Financial_Accuracy": [financial_accuracy],
    "Revenue_Growth": [average_growth],
    "Profit_Margin": [profit_margin],
    "DCF_Value": [dcf_value]
})

synthesis_features

,Stock_Accuracy,Sentiment_Accuracy,Risk_Accuracy,Market_Accuracy,Financial_Accuracy,Revenue_Growth,Profit_Margin,DCF_Value
0,0.665072,0.675791,0.870968,0.652893,0.75,19.688285,36.23393,7.868906e+11


In [28]:
investment_score = 0

if stock_signal == "UP":
    investment_score += 2
else:
    investment_score -= 2

if market_signal == "YES":
    investment_score += 2
else:
    investment_score -= 2

if average_growth > 0:
    investment_score += 2
else:
    investment_score -= 2

if profit_margin > 0:
    investment_score += 2
else:
    investment_score -= 2

In [29]:
risk_prediction_data = pd.read_csv(
    DATA_PATH + "nvidia_risk_predictions.csv"
)

high_risk_ratio = (
    risk_prediction_data["Predicted_High_Risk"]
    .mean()
)

if high_risk_ratio > 0.5:
    investment_score -= 2
else:
    investment_score += 2

In [30]:
if investment_score >= 5:
    investment_class = "BUY"
elif investment_score >= 2:
    investment_class = "HOLD"
else:
    investment_class = "SELL"

print("Investment Score:", investment_score)
print("Investment Recommendation:", investment_class)

Investment Score: 10
Investment Recommendation: BUY


In [31]:
synthesis_result = pd.DataFrame({
    "Stock_Signal": [stock_signal],
    "Market_Signal": [market_signal],
    "Revenue_Growth": [average_growth],
    "Profit_Margin": [profit_margin],
    "DCF_Enterprise_Value": [dcf_value],
    "High_Risk_Ratio": [high_risk_ratio],
    "Investment_Score": [investment_score],
    "Investment_Recommendation": [investment_class]
})

synthesis_result

,Stock_Signal,Market_Signal,Revenue_Growth,Profit_Margin,DCF_Enterprise_Value,High_Risk_Ratio,Investment_Score,Investment_Recommendation
0,UP,YES,19.688285,36.23393,7.868906e+11,0.225806,10,BUY


In [32]:
synthesis_result.to_csv(
    DATA_PATH + "investment_synthesis_result.csv",
    index=False
)

In [33]:
joblib.dump(
    {
        "type": "rule_based_synthesis",
        "features": synthesis_features.columns.tolist()
    },
    OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
)

['../outputs/nvidia_investment_synthesis_model.pkl']

In [34]:
model_registry["Investment Synthesis"] = {
    "model": "Rule-Based Synthesis Model",
    "path": OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
}

with open(
    OUTPUT_PATH + "model_registry.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_registry,
        f,
        indent=4
    )

In [35]:
for name in model_registry:
    print(name)

Stock Prediction
News Sentiment
Risk Classification
Market & Competitor
Valuation
Research
Investment Synthesis


In [37]:
synthesis_result

,Stock_Signal,Market_Signal,Revenue_Growth,Profit_Margin,DCF_Enterprise_Value,High_Risk_Ratio,Investment_Score,Investment_Recommendation
0,UP,YES,19.688285,36.23393,7.868906e+11,0.225806,10,BUY


In [38]:
synthesis_result = pd.read_csv(
    DATA_PATH + "investment_synthesis_result.csv"
)

synthesis_result

,Stock_Signal,Market_Signal,Revenue_Growth,Profit_Margin,DCF_Enterprise_Value,High_Risk_Ratio,Investment_Score,Investment_Recommendation
0,UP,YES,19.688285,36.23393,7.868906e+11,0.225806,10,BUY


In [39]:
investment_score_details = pd.DataFrame({
    "Factor": [
        "Stock Signal",
        "Market Position",
        "Revenue Growth",
        "Profit Margin",
        "Risk Level"
    ],
    "Result": [
        stock_signal,
        market_signal,
        "Positive" if average_growth > 0 else "Negative",
        "Positive" if profit_margin > 0 else "Negative",
        "High" if high_risk_ratio > 0.5 else "Low"
    ]
})

investment_score_details

,Factor,Result
0,Stock Signal,UP
1,Market Position,YES
2,Revenue Growth,Positive
3,Profit Margin,Positive
4,Risk Level,Low


In [40]:
investment_score_details.to_csv(
    DATA_PATH + "investment_score_details.csv",
    index=False
)

In [41]:
final_recommendation = {
    "Investment_Score": investment_score,
    "Recommendation": investment_class,
    "Stock_Signal": stock_signal,
    "Market_Position": market_signal,
    "Revenue_Growth": average_growth,
    "Profit_Margin": profit_margin,
    "High_Risk_Ratio": high_risk_ratio
}

final_recommendation

{'Investment_Score': 10,
 'Recommendation': 'BUY',
 'Stock_Signal': 'UP',
 'Market_Position': 'YES',
 'Revenue_Growth': 19.688285154308403,
 'Profit_Margin': 36.23393029649996,
 'High_Risk_Ratio': np.float64(0.22580645161290322)}

In [42]:
with open(
    OUTPUT_PATH + "final_investment_recommendation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_recommendation,
        f,
        indent=4
    )

In [43]:
model_registry["Investment Synthesis"] = {
    "model": "Rule-Based Synthesis",
    "path": OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
}

with open(
    OUTPUT_PATH + "model_registry.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_registry,
        f,
        indent=4
    )

In [44]:
print("Completed model components:", len(model_registry))

for name in model_registry:
    print(name)

Completed model components: 7
Stock Prediction
News Sentiment
Risk Classification
Market & Competitor
Valuation
Research
Investment Synthesis


In [45]:
print(
    "Final NVIDIA Investment Recommendation:",
    investment_class
)

print(
    "Investment Score:",
    investment_score
)

Final NVIDIA Investment Recommendation: BUY
Investment Score: 10


In [47]:
verification_df = research_df[
    ["text", "Topic_Name"]
].copy()

verification_df = verification_df.dropna(
    subset=["text"]
).reset_index(drop=True)

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
verification_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True
)

verification_tfidf = verification_vectorizer.fit_transform(
    verification_df["text"]
)

In [51]:
verification_df = research_df[
    ["text", "Topic_Name"]
].copy()

verification_df = verification_df.dropna(
    subset=["text"]
).reset_index(drop=True)

In [52]:
verification_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True
)

verification_tfidf = verification_vectorizer.fit_transform(
    verification_df["text"]
)

In [53]:
similarity_matrix = cosine_similarity(
    verification_tfidf
)

In [54]:
verification_pairs = []

for i in range(len(similarity_matrix)):
    for j in range(i + 1, len(similarity_matrix)):
        similarity = similarity_matrix[i, j]

        if similarity >= 0.80:
            verification_pairs.append({
                "Statement_1": verification_df.iloc[i]["text"],
                "Statement_2": verification_df.iloc[j]["text"],
                "Similarity": similarity
            })

verification_pairs_df = pd.DataFrame(
    verification_pairs
)

In [55]:
verification_df["Text_Length"] = (
    verification_df["text"]
    .astype(str)
    .str.len()
)

verification_df["Word_Count"] = (
    verification_df["text"]
    .astype(str)
    .str.split()
    .str.len()
)

In [56]:
verification_df["Verification_Flag"] = (
    (verification_df["Word_Count"] < 8)
    |
    (verification_df["Word_Count"] > 500)
).astype(int)

In [57]:
verification_df["Verification_Status"] = (
    verification_df["Verification_Flag"]
    .map({
        0: "PASS",
        1: "REVIEW"
    })
)

In [58]:
verification_df["Red_Team_Score"] = 100

verification_df.loc[
    verification_df["Verification_Status"] == "REVIEW",
    "Red_Team_Score"
] -= 30

In [59]:
if len(verification_pairs_df) > 0:
    duplicated_texts = set(
        verification_pairs_df["Statement_1"]
    ).union(
        set(verification_pairs_df["Statement_2"])
    )

    verification_df.loc[
        verification_df["text"].isin(duplicated_texts),
        "Red_Team_Score"
    ] -= 20

In [60]:
verification_df["Final_Status"] = np.where(
    verification_df["Red_Team_Score"] >= 70,
    "PASS",
    "REVIEW"
)

In [61]:
red_team_summary = pd.DataFrame({
    "Metric": [
        "Total Statements",
        "Passed",
        "Requires Review",
        "Potential Duplicate Pairs",
        "Average Red-Team Score"
    ],
    "Value": [
        len(verification_df),
        (verification_df["Final_Status"] == "PASS").sum(),
        (verification_df["Final_Status"] == "REVIEW").sum(),
        len(verification_pairs_df),
        verification_df["Red_Team_Score"].mean()
    ]
})

red_team_summary

,Metric,Value
0,Total Statements,1953.000000
1,Passed,1945.000000
2,Requires Review,8.000000
3,Potential Duplicate Pairs,244.000000
4,Average Red-Team Score,95.028162


In [62]:
verification_df.to_csv(
    DATA_PATH + "nvidia_red_team_verification.csv",
    index=False
)

verification_pairs_df.to_csv(
    DATA_PATH + "nvidia_potential_duplicate_claims.csv",
    index=False
)

red_team_summary.to_csv(
    DATA_PATH + "nvidia_red_team_summary.csv",
    index=False
)

In [63]:
joblib.dump(
    {
        "type": "rule_based_red_team_verification",
        "rules": [
            "short_or_long_statement",
            "duplicate_claim_detection"
        ]
    },
    OUTPUT_PATH + "nvidia_red_team_model.pkl"
)

['../outputs/nvidia_red_team_model.pkl']

In [64]:
joblib.dump(
    verification_vectorizer,
    OUTPUT_PATH + "nvidia_red_team_tfidf_vectorizer.pkl"
)

['../outputs/nvidia_red_team_tfidf_vectorizer.pkl']

In [65]:
model_registry["Red-Team Verification"] = {
    "model": "Rule-Based Verification Model",
    "path": OUTPUT_PATH + "nvidia_red_team_model.pkl"
}

with open(
    OUTPUT_PATH + "model_registry.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_registry,
        f,
        indent=4
    )

In [66]:
for name in model_registry:
    print(name)

Stock Prediction
News Sentiment
Risk Classification
Market & Competitor
Valuation
Research
Investment Synthesis
Red-Team Verification


In [67]:
red_team_summary

,Metric,Value
0,Total Statements,1953.000000
1,Passed,1945.000000
2,Requires Review,8.000000
3,Potential Duplicate Pairs,244.000000
4,Average Red-Team Score,95.028162


In [68]:
completed_models_df = pd.DataFrame({
    "Model_Component": list(model_registry.keys()),
    "Status": ["Completed"] * len(model_registry)
})

completed_models_df.to_csv(
    DATA_PATH + "completed_models.csv",
    index=False
)

In [69]:
for file in sorted(
    os.listdir(OUTPUT_PATH)
):
    if file.endswith(".pkl"):
        print(file)

best_nvidia_financial_model.pkl
best_nvidia_market_competitor_model.pkl
best_nvidia_risk_model.pkl
best_nvidia_stock_model.pkl
nvidia_balanced_random_forest_model.pkl
nvidia_dcf_model.pkl
nvidia_financial_scaler.pkl
nvidia_financial_sentiment_model.pkl
nvidia_investment_synthesis_model.pkl
nvidia_logistic_final_model.pkl
nvidia_logistic_scaler.pkl
nvidia_market_competitor_scaler.pkl
nvidia_random_forest_chronological.pkl
nvidia_random_forest_model.pkl
nvidia_red_team_model.pkl
nvidia_red_team_tfidf_vectorizer.pkl
nvidia_research_tfidf_vectorizer.pkl
nvidia_research_topic_model.pkl
nvidia_risk_logistic_model.pkl
nvidia_risk_tfidf_vectorizer.pkl
nvidia_sentiment_tfidf_vectorizer.pkl
nvidia_stock_chronological_scaler.pkl
nvidia_stock_logistic_model.pkl
nvidia_stock_scaler.pkl


In [70]:
model_registry

{'Stock Prediction': {'model': 'Logistic Regression',
  'path': '../outputs/nvidia_logistic_final_model.pkl'},
 'News Sentiment': {'model': 'Logistic Regression',
  'path': '../outputs/nvidia_financial_sentiment_model.pkl'},
 'Risk Classification': {'model': 'Random Forest',
  'path': '../outputs/best_nvidia_risk_model.pkl'},
 'Market & Competitor': {'model': 'Logistic Regression',
  'path': '../outputs/best_nvidia_market_competitor_model.pkl'},
 'Valuation': {'model': 'DCF + P/E Valuation',
  'path': '../outputs/nvidia_dcf_model.pkl'},
 'Research': {'model': 'TF-IDF + K-Means Topic Model',
  'path': '../outputs/nvidia_research_topic_model.pkl'},
 'Investment Synthesis': {'model': 'Rule-Based Synthesis',
  'path': '../outputs/nvidia_investment_synthesis_model.pkl'},
 'Red-Team Verification': {'model': 'Rule-Based Verification Model',
  'path': '../outputs/nvidia_red_team_model.pkl'}}

In [71]:
print(
    f"Total completed components: {len(model_registry)}"
)

Total completed components: 8
